In [ ]:
"""
PointVoxelFormer (Heinrich 2024) adaptado a KiTS23.
Híbrido: MLPs point-wise alternando con rasterización diferenciable + CNN 3D.
"""

import os, math, time, numpy as np, json
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.spatial import cKDTree
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

device = torch.device('cuda')
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# --- Configuración ---
DRIVE_BASE   = '/content/drive/MyDrive/kits2'
VOLUMES_DIR  = f'{DRIVE_BASE}/volumes_for_saliency'
CKPT_DIR     = f'{DRIVE_BASE}/checkpoints_pvf'
os.makedirs(CKPT_DIR, exist_ok=True)

NUM_POINTS   = 36_864
BATCH_SIZE   = 4
NUM_EPOCHS   = 300
LR           = 1e-3
WEIGHT_DECAY = 1e-2
NUM_CLASSES  = 3
GRID         = 48
HU_THRESH    = 0.225
KNN_K        = 3
CHUNK        = 300_000
SEED         = 42

FOPS = {1: 0.40, 2: 0.50, 3: 0.10}

np.random.seed(SEED)
torch.manual_seed(SEED)

# --- Dataset ---
class KiTSDataset(Dataset):
    def __init__(self, pt_files, num_points, mode='train'):
        self.files = pt_files; self.num_points = num_points; self.mode = mode

    def __len__(self):
        return len(self.files)

    def _sample(self, coords, hu, labels):
        if self.mode == 'train':
            available = {}
            for c in FOPS.keys():
                ix = np.where(labels == c)[0]
                if len(ix) > 0: available[c] = ix
            if not available:
                idx = np.random.choice(len(labels), self.num_points, replace=True)
            else:
                total_r = sum(FOPS[c] for c in available)
                sel = []; assigned = 0; cls_list = list(available.keys())
                for i, c in enumerate(cls_list):
                    n = self.num_points-assigned if i==len(cls_list)-1 else int(self.num_points*FOPS[c]/total_r)
                    if i < len(cls_list)-1: assigned += n
                    sel.append(np.random.choice(available[c], n, replace=True))
                idx = np.concatenate(sel)
        else:
            fg_idx = np.where(labels > 0)[0]
            if len(fg_idx) < 10: fg_idx = np.arange(len(labels))
            idx = np.random.choice(fg_idx, self.num_points, replace=True)

        if len(idx) > self.num_points:
            idx = idx[:self.num_points]
        elif len(idx) < self.num_points:
            idx = np.concatenate([idx, np.random.choice(idx, self.num_points-len(idx), replace=True)])
        np.random.shuffle(idx)
        return coords[idx], hu[idx], labels[idx]

    def __getitem__(self, i):
        data = torch.load(self.files[i], weights_only=False)
        vol = data['volume'].squeeze().numpy().astype(np.float32)
        lab = data['label'].numpy().astype(np.int64)
        D, H, W = vol.shape
        dz = np.linspace(0,1,D,dtype=np.float32); dy = np.linspace(0,1,H,dtype=np.float32); dx = np.linspace(0,1,W,dtype=np.float32)
        zz, yy, xx = np.meshgrid(dz,dy,dx,indexing='ij')
        coords = np.stack([xx.ravel(),yy.ravel(),zz.ravel()],axis=1)
        hu = vol.ravel(); labels = lab.ravel()

        coords, hu, labels = self._sample(coords, hu, labels)
        c = coords - coords.mean(0,keepdims=True)
        c = c / (np.max(np.linalg.norm(c,axis=1))+1e-6)
        labels = labels - 1

        if self.mode == 'train':
            angle = np.random.uniform(0,2*math.pi)
            rot = np.array([[math.cos(angle),-math.sin(angle),0],[math.sin(angle),math.cos(angle),0],[0,0,1]],dtype=np.float32)
            c = c @ rot.T
            c += np.random.normal(0,0.002,c.shape).astype(np.float32)
            c *= np.random.uniform(0.9,1.1)

        return torch.from_numpy(c).float(), torch.from_numpy(hu[:,None]).float(), torch.from_numpy(labels).long()

# --- Rasterización Diferenciable ---
def normalize_to_grid(xyz, grid):
    mn = xyz.amin(dim=1, keepdim=True)
    mx = xyz.amax(dim=1, keepdim=True)
    xyz01 = (xyz - mn) / (mx - mn + 1e-6)
    return xyz01 * (grid - 1)

def trilinear_splat(xyz, feat, grid):
    B, N, C = feat.shape
    dev = feat.device
    x0 = torch.floor(xyz).long().clamp(0, grid-2)
    xf = xyz - x0.float()

    vox = torch.zeros(B, C, grid, grid, grid, device=dev)
    weight = torch.zeros(B, 1, grid, grid, grid, device=dev)

    for dx_ in (0,1):
        for dy_ in (0,1):
            for dz_ in (0,1):
                wx = xf[...,0] if dx_ else (1-xf[...,0])
                wy = xf[...,1] if dy_ else (1-xf[...,1])
                wz = xf[...,2] if dz_ else (1-xf[...,2])
                w = (wx*wy*wz).unsqueeze(-1)
                ix = (x0[...,0]+dx_).clamp(0,grid-1)
                iy = (x0[...,1]+dy_).clamp(0,grid-1)
                iz = (x0[...,2]+dz_).clamp(0,grid-1)
                flat = (ix*grid*grid + iy*grid + iz)
                wf = (w*feat).permute(0,2,1)
                vox.view(B,C,-1).scatter_add_(2, flat.unsqueeze(1).expand(-1,C,-1), wf)
                weight.view(B,1,-1).scatter_add_(2, flat.unsqueeze(1), w.permute(0,2,1))
    vox = vox / (weight + 1e-5)
    return vox

def trilinear_sample(vox, xyz, grid):
    B = vox.shape[0]; N = xyz.shape[1]
    g = (xyz / (grid-1)) * 2 - 1
    g = g.view(B, N, 1, 1, 3)
    g = g[..., [2,1,0]]
    out = F.grid_sample(vox, g, mode='bilinear', align_corners=True)
    return out.view(B, vox.shape[1], N).permute(0,2,1)

# --- Modelos ---
class PVFBlock(nn.Module):
    def __init__(self, C, grid):
        super().__init__()
        self.grid = grid
        self.norm1 = nn.LayerNorm(C)
        self.mlp = nn.Sequential(nn.Linear(C, C*2), nn.GELU(), nn.Linear(C*2, C))
        self.norm2 = nn.LayerNorm(C)
        self.conv = nn.Sequential(
            nn.Conv3d(C, C, 3, padding=1, bias=False), nn.BatchNorm3d(C), nn.GELU(),
            nn.Conv3d(C, C, 3, padding=1, bias=False), nn.BatchNorm3d(C))

    def forward(self, xyz_grid, feat):
        feat = feat + self.mlp(self.norm1(feat))
        vox = trilinear_splat(xyz_grid, self.norm2(feat), self.grid)
        vox = self.conv(vox)
        sampled = trilinear_sample(vox, xyz_grid, self.grid)
        return feat + sampled

class PointVoxelFormer(nn.Module):
    def __init__(self, in_feat=1, num_classes=3, C=64, grid=48, n_blocks=4):
        super().__init__()
        self.grid = grid
        self.stem = nn.Sequential(nn.Linear(in_feat+3, C), nn.GELU(), nn.Linear(C, C))
        self.blocks = nn.ModuleList([PVFBlock(C, grid) for _ in range(n_blocks)])
        self.head = nn.Sequential(nn.LayerNorm(C), nn.Linear(C, C), nn.GELU(),
                                   nn.Dropout(0.3), nn.Linear(C, num_classes))

    def forward(self, xyz, feat):
        xyz_grid = normalize_to_grid(xyz, self.grid)
        x = self.stem(torch.cat([xyz, feat], dim=-1))
        for blk in self.blocks:
            x = blk(xyz_grid, x)
        return self.head(x).transpose(1,2)

# --- Splits de Datos ---
np.random.seed(SEED)
all_files = sorted([os.path.join(VOLUMES_DIR,f) for f in os.listdir(VOLUMES_DIR) if f.endswith('.pt')])
np.random.shuffle(all_files)
n = len(all_files)
train_f = all_files[:int(n*0.8)]
val_f = all_files[int(n*0.8):int(n*0.9)]
test_f = all_files[int(n*0.9):]

print(f"Train:{len(train_f)} Val:{len(val_f)} Test:{len(test_f)}")

train_dl = DataLoader(KiTSDataset(train_f, NUM_POINTS, 'train'), batch_size=BATCH_SIZE,
                      shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
val_dl = DataLoader(KiTSDataset(val_f, NUM_POINTS, 'val'), batch_size=1, shuffle=False, num_workers=0)

# --- Instanciación ---
model = PointVoxelFormer(in_feat=1, num_classes=NUM_CLASSES, C=64, grid=GRID, n_blocks=4).to(device)
print(f"Parámetros: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

weights = torch.tensor([0.121, 0.096, 1.000], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)

def dice_pc(pred, gt, nc=3):
    p, g = pred.cpu().numpy(), gt.cpu().numpy()
    out = []
    for c in range(nc):
        pi = (p==c).astype(float); gi = (g==c).astype(float)
        i = (pi*gi).sum(); u = pi.sum()+gi.sum()
        out.append(0. if u==0 else 2*i/(u+1e-8))
    return out

start_ep = 0; best_fg = 0.0
last_ckpt = f'{CKPT_DIR}/last.pth'; best_ckpt = f'{CKPT_DIR}/best.pth'

if os.path.exists(last_ckpt):
    ck = torch.load(last_ckpt, weights_only=False)
    model.load_state_dict(ck['model_state_dict'])
    optimizer.load_state_dict(ck['optimizer_state_dict'])
    start_ep = ck['epoch']+1; best_fg = ck.get('best_fg',0.)
    print(f"✓ Resume epoch {start_ep}")

# --- Entrenamiento ---
for ep in range(start_ep, NUM_EPOCHS):
    t0 = time.time(); model.train(); tl = 0.
    for xyz, feat, lbl in tqdm(train_dl, desc=f"E{ep+1:03d}", leave=False):
        xyz, feat, lbl = xyz.to(device), feat.to(device), lbl.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xyz,feat), lbl)
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        tl += loss.item()
    tl /= len(train_dl)

    model.eval(); vl = 0.; vd = []
    with torch.no_grad():
        for xyz, feat, lbl in val_dl:
            xyz, feat, lbl = xyz.to(device), feat.to(device), lbl.to(device)
            out = model(xyz,feat); vl += criterion(out,lbl).item()
            vd.append(dice_pc(out.argmax(1).squeeze(0), lbl.squeeze(0)))

    vl /= len(val_dl); vd = np.array(vd)
    vK, vT, vC = vd[:,0].mean(), vd[:,1].mean(), vd[:,2].mean(); vFG = (vK+vT+vC)/3
    scheduler.step()

    print(f"E{ep+1:03d} TL:{tl:.4f} VL:{vl:.4f} K:{vK:.4f} T:{vT:.4f} C:{vC:.4f} FG:{vFG:.4f} ({time.time()-t0:.0f}s)")
    if vFG > best_fg:
        best_fg = vFG
        torch.save({'epoch':ep,'model_state_dict':model.state_dict(),'val_fg':vFG,'val_tumor':vT,'val_kidney':vK}, best_ckpt)
        print(f"  ✓ Best FG={vFG:.4f} T={vT:.4f}")
    torch.save({'epoch':ep,'model_state_dict':model.state_dict(),'optimizer_state_dict':optimizer.state_dict(),'best_fg':best_fg}, last_ckpt)

# --- Evaluación ---
print("\n" + "-"*65)
ck = torch.load(best_ckpt, weights_only=False)
model.load_state_dict(ck['model_state_dict']); model.eval()
print(f"Best epoch {ck['epoch']+1} | Val FG={ck['val_fg']:.4f}")

print("Cargando test en RAM...")
test_data = {}
for f in test_f:
    try:
        d = torch.load(f, weights_only=False)
        test_data[os.path.basename(f)] = {'volume':d['volume'].squeeze().numpy().astype(np.float32),'label':d['label'].numpy().astype(np.int64)}
    except Exception as e: print(f"⚠ {f}: {e}")
print(f"✓ {len(test_data)} casos")

def eval_knn(vol_np, label_np):
    D, H, W = vol_np.shape
    dz = np.linspace(0,1,D,dtype=np.float32); dy = np.linspace(0,1,H,dtype=np.float32); dx = np.linspace(0,1,W,dtype=np.float32)
    zz, yy, xx = np.meshgrid(dz,dy,dx,indexing='ij')
    coords = np.stack([xx.ravel(),yy.ravel(),zz.ravel()],axis=1)
    hu = vol_np.ravel().astype(np.float32); labels = label_np.ravel().astype(np.int64)

    fg_idx = np.where(hu>HU_THRESH)[0]
    if len(fg_idx) < 10: fg_idx = np.arange(len(hu))
    sg = fg_idx[np.random.choice(len(fg_idx),NUM_POINTS,replace=True)]
    cs = coords[sg].copy(); hs = hu[sg,None]
    cn = cs-cs.mean(0,keepdims=True); cn = cn/(np.max(np.linalg.norm(cn,axis=1))+1e-6)

    xyz_t = torch.from_numpy(cn).float().unsqueeze(0).to(device)
    feat_t = torch.from_numpy(hs).float().unsqueeze(0).to(device)
    with torch.no_grad():
        pred_s = model(xyz_t,feat_t).argmax(1).squeeze(0).cpu().numpy()+1

    tree = cKDTree(cs); pred_fg = np.zeros(len(fg_idx),dtype=np.int64); fg_c = coords[fg_idx]
    for s in range(0,len(fg_idx),CHUNK):
        e = min(s+CHUNK,len(fg_idx)); _, ki = tree.query(fg_c[s:e],k=KNN_K,workers=-1)
        v = pred_s[ki]
        for i in range(e-s):
            vals, cnt = np.unique(v[i],return_counts=True); pred_fg[s+i] = vals[cnt.argmax()]

    pred_full = np.zeros(len(labels),dtype=np.int64); pred_full[fg_idx] = pred_fg
    pred_36k = pred_s-1; lbl_36k = np.clip(labels[sg]-1,0,2)

    d36 = {}
    for c_, nm in enumerate(['kidney','tumor','cyst']):
        p = (pred_36k==c_).astype(float); g = (lbl_36k==c_).astype(float); i = (p*g).sum(); u = p.sum()+g.sum()
        d36[nm] = float('nan') if u==0 else float(2*i/(u+1e-8))

    dvol = {}
    for c_, nm in enumerate(['bg','kidney','tumor','cyst']):
        p = (pred_full==c_).astype(float); g = (labels==c_).astype(float); i = (p*g).sum(); u = p.sum()+g.sum()
        dvol[nm] = float('nan') if u==0 else float(2*i/(u+1e-8))

    return d36, dvol

r36, rvol = [], []
for fn, dd in tqdm(test_data.items(),desc="Test"):
    try:
        d36, dvol = eval_knn(dd['volume'],dd['label']); r36.append(d36); rvol.append(dvol)
        tqdm.write(f"  {fn}: 36K T={d36['tumor']:.3f} | Vol T={dvol['tumor']:.3f}")
    except Exception as e: tqdm.write(f"⚠ {fn}: {e}")

def st(r,ks): return {k:(np.nanmean([x[k] for x in r]),np.nanstd([x[k] for x in r])) for k in ks}
s36 = st(r36,['kidney','tumor','cyst']); sv = st(rvol,['kidney','tumor','cyst'])

print("\n" + "-"*60)
print("PointVoxelFormer - RESULTADOS")
print("-"*60)
print(f"{'Métrica':<10} {'36K uniform':>16} {'Volumen completo':>18}")
for nm in ['kidney','tumor','cyst']:
    print(f"{nm:<10} {s36[nm][0]:>8.4f}±{s36[nm][1]:.3f}  {sv[nm][0]:>9.4f}±{sv[nm][1]:.3f}")
print(f"\nTumor Vol: {sv['tumor'][0]:.4f} | Nowakowski: 0.693 | {'✅' if sv['tumor'][0]>0.693 else '❌'}")

json.dump({
    'pvf_36k': {k:{'mean':float(s36[k][0]),'std':float(s36[k][1])} for k in s36},
    'pvf_vol': {k:{'mean':float(sv[k][0]),'std':float(sv[k][1])} for k in sv},
    'best_epoch': int(ck['epoch']+1), 'n': len(r36)
}, open(f'{CKPT_DIR}/results.json','w'), indent=2)

print(f"✓ Guardado")